# Semenov et al. (2026) from dandiset 001525: membrane recovery after 1-us electroporation

**Paper.** Semenov, Silkuniene, Silkunas, Pakhomova, Bixler, Ibey, Kochunov & Pakhomov (2026),
"Sub-microsecond optical measurements of cell membrane charging and lesioning by pulsed electric
fields", Bioelectrochemistry 171, 109283, doi:10.1016/j.bioelechem.2026.109283.

**Dataset.** [DANDI:001525](https://dandiarchive.org/dandiset/001525/0.261006.1856), version
`0.261006.1856` (doi:10.48324/dandi.001525/0.261006.1856), streamed from the archive. The paper
cites no version (its data statement says data are available on request); this is the most
recent and only published version, and its content (73 files, 2.51 GB) is the same as the draft's.

**Claims file.** `claims/001525/10.1016-j.bioelechem.2026.109283.yaml`, written from the version of
record (main text, Methods, legends; no supplement).

**What the dataset holds.** Ten CHO cells loaded with Di-8-ANEPPS, 73 sets of 100 strobe images:
per cell a single 50 V pulse and conditioning pulses of 100 to 350 V each followed by a 50 V test
pulse, after 1 us (cells c1 to c4) or 5 us (c5 to c10). This is the two-pulse protocol of
Section 3.7 and Fig. 7A and 7B (bottom). Nothing else of the paper is here.

| Claim | Tested here |
|---|---|
| fig7a-test-pulse-cosine-up-to-1000 | tested |
| fig7b-test-pulse-unchanged-to-760 | tested |
| fig7b-test-pulse-reduced-1200-1400 | tested |
| fig5b-conductivity-calibration | tested (a computation from Eq. 2, no recording) |
| fig7b-conditioning-pulse-below-linear | blocked, data missing (Fig. 4B images) |
| fig7c-conductivity-drops-10-fold, fig7c-full-reseal-mild-residual-strong | blocked, depend on the claim above |
| fig1, fig2 (6 claims), fig3 (2), fig4 (6), fig5a (2), fig5c, fig6 (2) | blocked, data missing |


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
import strobe as S

assets = S.list_assets()
meta = S.read_all_meta(assets)
subjects = sorted({d["subject"] for d in meta}, key=lambda s: int(s.split("_c")[1]))
print(f"dandiset {S.DANDISET_ID} version {S.DANDISET_VERSION}: {len(assets)} assets, "
      f"{sum(a['size'] for a in assets)/1e9:.2f} GB, {len(subjects)} subjects")

In [ ]:
# Data check: what the files hold, against the paper's description
df = pd.DataFrame([{k: d[k] for k in ("subject", "kind", "cond_v", "test_v", "interval_us",
                     "shape", "rate", "unit", "electrode_gap", "dye")} for d in meta])
df["shape"] = df["shape"].astype(str)
print("files per subject and protocol (conditioning volts; 'single' = one 50 V pulse):")
tab = df.assign(level=df.cond_v.fillna(0).astype(int).astype(str).replace("0", "single"))
print(pd.crosstab([tab.subject, tab.interval_us.fillna(0).astype(int)], tab.level).to_string())
print("\nimage stack shapes:", Counter(df["shape"]))
print("frame rate (Hz):", Counter(df.rate), "| data unit:", Counter(df.unit))
print("test / single pulse volts:", Counter(df.test_v), "| interval (us):", Counter(df.interval_us.dropna()))
print("electrode gap stated:", Counter(df.electrode_gap), "(paper, Methods 2.2: spacing 1.6 mm)")
print("dye:", Counter(df.dye))
print("\nseries description:", meta[0]["series_description"])
print("\ngeneral description in every file (identical:",
      len({d['description'].split('####')[-1] for d in meta}) == 1, "):")
print(meta[0]["description"].split("#####")[-1][:700])

In [ ]:
# Load and reduce every file: cell, trace, collapsed images, dF profiles under each reading.
results = S.process_subjects(meta, subjects, workers=10)
five = [s for s in subjects if all(o["interval_us"] in (None, 5) for o in results[s])]
one = [s for s in subjects if s not in five]
print("5-us cells (Fig. 7):", five)
print("1-us cells (not in the paper):", one)
# Recordings deposited twice: compare a digest of each image stack
dig = Counter(round(o["data_digest"]) for s in subjects for o in results[s])
print("files:", sum(len(results[s]) for s in subjects), "| repeated image stacks:",
      sum(v - 1 for v in dig.values() if v > 1))
rows = []
for s in subjects:
    for o in results[s]:
        rows.append(dict(subject=s.split("_")[-1], level=S.level_of(o), cx=o["cx"], cy=o["cy"],
                         r_px=o["r_membrane"], n_dark=o["n_dark"], dark=str(o["dark_idx"])))
cells = pd.DataFrame(rows)
print("\ncell position (px) and membrane radius, range over a subject's files:")
print(cells.groupby("subject", sort=False).agg(cx_min=("cx", "min"), cx_max=("cx", "max"),
      cy_min=("cy", "min"), cy_max=("cy", "max"), r_min=("r_px", "min"), r_max=("r_px", "max"),
      dark_frames=("n_dark", "sum")).round(1).to_string())
print("\ndark (no-flash) frames per file:", Counter(cells.n_dark), "| at frame index:",
      Counter(i for s in subjects for o in results[s] for i in o["dark_idx"]).most_common(8))
fm = np.array([o["frame_mean"] for s in subjects for o in results[s]])
dk = np.concatenate([S.dark_frames(o["frame_mean"][:, None, None]) for s in subjects for o in results[s]])
print(f"field mean of dark frames {fm.ravel()[dk].mean():.0f} counts, of lit frames "
      f"{fm.ravel()[~dk].mean():.0f}; pixel range "
      f"{min(o['frame_mean'].min() for s in subjects for o in results[s]):.0f} to "
      f"{max(o['frame_mean'].max() for s in subjects for o in results[s]):.0f} (field means)")

In [ ]:
# Field direction and clock. The files do not give the field direction in image coordinates;
# every file's profile is fit with a cosine and the phases are pooled (5-us cells).
MAIN = ("dF", True, True)          # dark frames dropped, annulus background (see below)
READINGS = {("dF", d, b): f"dark {'dropped' if d else 'kept'}, background {'annulus' if b else 'none'}"
            for d in (False, True) for b in (False, True)}
th = np.arange(360) + 0.5
res5 = {s: results[s] for s in five}
PHI = {}
for key, name in READINGS.items():
    PHI[key], phis = S.field_angle(res5, key)
    print(f"{name}: depolarized pole at image angle {PHI[key]:.1f} deg, per-file phases "
          f"{np.min(phis):.0f} to {np.max(phis):.0f} (sd {np.std(phis):.1f})")
phi = PHI[MAIN]
# Clock: the first 40 images are baseline and the next 60 carry the pulse (Methods 2.3). Per file,
# the pole contrast (cathodic minus anodic trace intensity over their mean) is split at every
# frame k; the split with the largest t statistic should be 40.
best, tmax = [], []
for s in subjects:
    for o in results[s]:
        ft = np.array([S.rotate_to_field(f, phi) for f in o["frame_trace"]])
        c, a = np.array([S.pole_means(f) for f in ft]).T
        ok = ~S.dark_frames(o["frame_mean"][:, None, None])
        con = (c - a) / ((c + a) / 2)
        ts = []
        for k in range(10, 91):
            x0, x1 = con[:k][ok[:k]], con[k:][ok[k:]]
            ts.append(stats.ttest_ind(x1, x0).statistic)
        best.append(10 + int(np.argmax(ts))); tmax.append(np.max(ts))
best = np.array(best)
print(f"\nbest split frame over {len(best)} files: at 40 in {np.sum(best == 40)}, within 39 to 41 in "
      f"{np.sum(abs(best - 40) <= 1)}; others {sorted(best[abs(best - 40) > 1].tolist())}")
print(f"t statistic at the best split: median {np.median(tmax):.1f}, min {np.min(tmax):.1f}")
print("The series description says exposures began 1 us after the start of recording; the images "
      "show the pulse from frame 40 (6.7 s at 6 Hz).")

In [ ]:
# Polarization scale under each reading, against the paper's: Fig. 7A shows the 150/150 profile
# at about +-13% dF, and Fig. 2C at about +-11% at 150 V/cm; Results 3.3 give Di-8-ANEPPS 9.1% dF
# per 100 mV, and 1.5 E R at 150 V/cm and R = 8 um is 180 mV, about 16% dF.
for key, name in READINGS.items():
    P = S.profiles_by_level(results, five, key, PHI[key])
    A, ph, r2, _ = S.cosine_fit(th, P["single"].mean(0))
    off = P["single"].mean()
    print(f"{name}: 150/150 cosine amplitude {A:.1f}% dF, profile mean {off:.2f}% dF")
print("Background (annulus) in counts, baseline image, 5-us files: ",
      np.round(np.percentile([o[('bg', True, True)][0] for s in five for o in results[s]], [0, 50, 100])))
print("Trace intensity (baseline, mean over perimeter), counts: ",
      np.round(np.percentile([o['frame_trace'][1:40].mean() for s in five for o in results[s]], [0, 50, 100])))

**Readings used throughout.** Three details the paper does not fix could move the numbers, and
every claim below is shown under each:

- background: (a) none, as Methods 2.3 describe for the potentiometric images; (b) the median of
  an annulus 1.6 to 2.2 cell radii from the centre, a cell-free region as Methods 2.4 use for Fluo-4.
- dark frames: (a) kept, (b) dropped. Images with no laser flash are in most files (frame 0).
- the cell's control (150/150) when a cell has two single-pulse files: their mean, the first, the last.
- the cells: all six 5-us cells (the paper has n = 5 and does not say which), and each set of five.

The main reading, adopted as an assumption, is background (b) with dark frames dropped: the
paper's own scale (Fig. 7A about 13% dF at 150/150, and 9.1% dF per 100 mV with a 150 V/cm IMP
near 180 mV) is met only with a background removed (output above), and an image with no flash
holds no fluorescence. The conditioning strengths are mapped to the dataset's volts by rank
(`STRENGTH_V_CM`), an assumption stated in the helper.

In [ ]:
def poles_for(key, subs=None, benchmark="mean"):
    subs = subs or five
    return S.pole_values(S.profiles_by_level(results, subs, key, PHI[key], benchmark))

def pct_of_control(poles, L, pole):
    return 100 * S.polarization(poles, L, pole).mean() / S.polarization(poles, "single", pole).mean()

def test_vs_control(poles, L, pole, kind):
    x, c = S.polarization(poles, L, pole), S.polarization(poles, "single", pole)
    if kind == "paired":
        return stats.ttest_rel(x, c).pvalue
    if kind == "unpaired":
        return stats.ttest_ind(x, c).pvalue
    return stats.ttest_1samp(100 * x / c, 100).pvalue   # variant: per-cell normalized, one-sample

POLES = ("cathodic", "anodic", "mean")
TESTS = ("paired", "unpaired")
VARIANTS = [(key, b, None) for key in READINGS for b in ("mean", "first", "last")] + \
           [(MAIN, "mean", tuple(s for s in five if s != drop)) for drop in five]
def vname(v):
    key, b, subs = v
    cells_ = "6 cells" if subs is None else "without " + [s for s in five if s not in subs][0].split("_")[-1]
    return f"{READINGS[key]}; control {b}; {cells_}"
P0 = poles_for(MAIN)
print("Main reading: per-cell pole dF (%) by conditioning strength (V/cm)")
for L in S.LEVELS:
    c, a = P0[L]
    print(f"{S.STRENGTH_V_CM[L]:>5}: cathodic {np.round(c, 1)}  anodic {np.round(a, 1)}")

## fig7a-test-pulse-cosine-up-to-1000

> After a conditioning pulse of up to 1000 V/cm, polarization by the 150 V/cm test pulse 5 us
> later still follows a cosine of the angle to the field, while 1200 and 1400 V/cm conditioning
> pulses markedly reduce charging at the cell poles.

Per strength the mean profile over cells is fit with A cos(theta - phi). Reading (a): at each pole
the fit lies within the s.e. of the mean pole dF. Reading (b): pole polarization does not differ
from the benchmark by the two-sided t-test of Fig. 7B (paired or unpaired). Hindered at 1200 and
1400 V/cm: pole polarization below the benchmark at p < 0.05 by that test. Pole readings as in
Fig. 7B: cathodic, anodic, mean of both.

In [ ]:
def fig7a_eval(key, subs=None, benchmark="mean", verbose=False):
    subs = subs or five
    prof = S.profiles_by_level(results, subs, key, PHI[key], benchmark)
    poles = S.pole_values(prof)
    fitinfo, band_ok = {}, {}
    for L in S.LEVELS:
        mp = prof[L].mean(0)
        A, ph, r2, fit = S.cosine_fit(th, mp)
        fc, fa = S.pole_means(fit)
        c, a = poles[L]
        sec, sea = c.std(ddof=1) / np.sqrt(len(c)), a.std(ddof=1) / np.sqrt(len(a))
        okc, oka = abs(c.mean() - fc) <= sec, abs(a.mean() - fa) <= sea
        fitinfo[L] = (A, ph, r2, c.mean() - fc, sec, a.mean() - fa, sea)
        band_ok[L] = {"cathodic": okc, "anodic": oka, "mean": okc and oka}
    low = [L for L in S.LEVELS if S.STRENGTH_V_CM[L] <= 1000]
    high = [300, 350]
    out = {}
    for pole in POLES:
        for test in TESTS:
            hind = all(test_vs_control(poles, L, pole, test) < 0.05 and
                       S.polarization(poles, L, pole).mean() < S.polarization(poles, "single", pole).mean()
                       for L in high)
            cos_a = all(band_ok[L][pole] for L in low)
            cos_b = all(test_vs_control(poles, L, pole, test) >= 0.05 for L in low[1:])
            out[(pole, test)] = (cos_a, cos_b, hind)
    if verbose:
        print("strength  A(%dF)  phase  R2    cath: mean-fit  s.e. | anod: mean-fit  s.e.")
        for L in S.LEVELS:
            A, ph, r2, dc, sec, da, sea = fitinfo[L]
            print(f"{S.STRENGTH_V_CM[L]:>6} {A:6.2f} {ph:6.1f} {r2:5.2f}   {dc:8.2f} {sec:5.2f}   | {da:8.2f} {sea:5.2f}")
    return out, fitinfo

out, fitinfo = fig7a_eval(MAIN, verbose=True)
print("\nMain reading: (pole, test): follows cosine by (a) s.e. band, (b) no difference; hindered at 1200 and 1400")
for k, v in out.items():
    print(k, v, "-> holds under (a)" if v[0] and v[2] else "", "-> holds under (b)" if v[1] and v[2] else "")
P = P0
print("\np values against the benchmark, main reading:")
for pole in POLES:
    print(pole, {S.STRENGTH_V_CM[L]: (round(test_vs_control(P, L, pole, 'paired'), 4),
                 round(test_vs_control(P, L, pole, 'unpaired'), 4)) for L in S.LEVELS[1:]})

In [ ]:
fig, ax = plt.subplots(1, 7, figsize=(17, 2.8), sharey=True)
prof = S.profiles_by_level(results, five, MAIN, PHI[MAIN])
bench = S.cosine_fit(th, prof["single"].mean(0))[3]
lo_, hi_ = np.inf, -np.inf
for i, L in enumerate(S.LEVELS):
    m, se = prof[L].mean(0), prof[L].std(0, ddof=1) / np.sqrt(len(five))
    ax[i].fill_between(th, m - se, m + se, color="0.8"); ax[i].plot(th, m, "k", lw=0.6)
    ax[i].plot(th, S.cosine_fit(th, m)[3], "C1"); ax[i].plot(th, bench, "C1--", lw=0.8)
    ax[i].set_title(f"{S.STRENGTH_V_CM[L]}/150"); ax[i].set_xticks([0, 180, 360])
    lo_, hi_ = min(lo_, (m - se).min()), max(hi_, (m + se).max())
ax[0].set_ylabel("dF, % (main reading)"); ax[3].set_xlabel("angle to the field, degrees")
plt.show()
print(f"plotted mean +- s.e. ranges from {lo_:.1f} to {hi_:.1f}% dF; all points within the axes")

In [ ]:
# Every reading
rows = []
for v in VARIANTS:
    key, b, subs = v
    o, fi = fig7a_eval(key, list(subs) if subs else None, b)
    holds_a = any(x[0] and x[2] for x in o.values())
    holds_b = any(x[1] and x[2] for x in o.values())
    rows.append(dict(variant=vname(v), holds_a=holds_a, holds_b=holds_b,
                     cos_a_any=any(x[0] for x in o.values()), cos_b_any=any(x[1] for x in o.values()),
                     hindered_any=any(x[2] for x in o.values()),
                     A_150=round(fi["single"][0], 1), A_1000=round(fi[250][0], 1), A_1400=round(fi[350][0], 1),
                     R2_1000=round(fi[250][2], 2), R2_1400=round(fi[350][2], 2)))
t7a = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70)
print(t7a.to_string())
print("\nclaim holds (under a pole, test and cosine reading) in", int((t7a.holds_a | t7a.holds_b).sum()),
      "of", len(t7a), "variants")

**Result: reproduced under the main reading, an assumption, and fragile.** With all six 5-us
cells, background removed and dark frames dropped, the mean polarization of the two poles does not
differ from the 150/150 benchmark after 360 to 1000 V/cm conditioning (paired p 0.27 to 0.39) and is
below it after 1200 V/cm (p = 0.039) and 1400 V/cm (p = 0.012), so the claim holds under reading (b)
with the paired test. The cosine amplitude falls from 10.9% (benchmark) to 9.4% at 1000 V/cm and
4.8% at 1400 V/cm. Reading (a), the s.e. band, fails at every pole: the profiles carry a uniform
negative offset (profile mean -4.6% at the benchmark) that A cos(theta - phi) has no term for, and
the fit misses the cathodic pole mean by 4.7% against an s.e. of 2.8% already at the benchmark.
Over the 18 variants the claim holds in 9; the cosine part (reading b) holds under some pole and
test in all 18, and what fails is the 1200 V/cm hindrance, which reaches p < 0.05 in none of the
five-cell subsets except the one without c5.

## fig7b-test-pulse-unchanged-to-760

> Polarization by the test pulse 6 us after the conditioning pulse does not differ from the
> undamaged control for conditioning pulses up to 760 V/cm, and is at about 80% of control after
> 1000 V/cm, a difference that is not significant.

A claim of absence on the paper's two-sided t-test, n = 5 (here 6), each pole, readings (i) paired,
(ii) unpaired; the one-sample test of per-cell normalized values is a variant. The 80% is the group
mean at 1000 V/cm over the group mean of the control, per pole reading.

In [ ]:
def unchanged_eval(poles):
    res = {}
    for pole in ("cathodic", "anodic"):
        for test in TESTS + ("one-sample",):
            res[(pole, test)] = {S.STRENGTH_V_CM[L]: test_vs_control(poles, L, pole, test)
                                 for L in (100, 150, 200, 250)}
    pct = {pole: pct_of_control(poles, 250, pole) for pole in POLES}
    return res, pct

r, pct = unchanged_eval(P0)
print("Main reading, p values against the control (360, 570, 760, 1000 V/cm):")
for k, v in r.items():
    print(f"{k[0]:>9} {k[1]:>10}:", {s: round(p, 4) for s, p in v.items()},
          "-> none below 0.05" if min(v.values()) >= 0.05 else "-> differs")
print("\nPercent of control (group means), all strengths:")
for pole in POLES:
    print(f"{pole:>9}:", {S.STRENGTH_V_CM[L]: round(pct_of_control(P0, L, pole), 1) for L in S.LEVELS})
print("\nEffect sizes, mean per-cell difference from control (% dF) and paired Cohen's d:")
for pole in ("cathodic", "anodic"):
    for L in (100, 150, 200, 250):
        dlt = S.polarization(P0, L, pole) - S.polarization(P0, "single", pole)
        print(f"  {pole} {S.STRENGTH_V_CM[L]}: {dlt.mean():+.2f} (s.e. {dlt.std(ddof=1)/np.sqrt(len(dlt)):.2f}), d = {dlt.mean()/dlt.std(ddof=1):+.2f}")

In [ ]:
rows = []
for v in VARIANTS:
    key, b, subs = v
    poles = poles_for(key, list(subs) if subs else None, b)
    r, pct = unchanged_eval(poles)
    row = dict(variant=vname(v))
    for test in TESTS:
        row[f"absent_{test}"] = all(min(r[(pole, test)].values()) >= 0.05 for pole in ("cathodic", "anodic"))
        row[f"min_p_{test}"] = round(min(min(r[(pole, test)].values()) for pole in ("cathodic", "anodic")), 4)
    row["absent_one_sample"] = all(min(r[(pole, "one-sample")].values()) >= 0.05 for pole in ("cathodic", "anodic"))
    row.update({f"pct1000_{p}": round(pct[p], 1) for p in POLES})
    rows.append(row)
t7b = pd.DataFrame(rows)
print(t7b.to_string())
print("\nabsence holds under both test readings in", int((t7b.absent_paired & t7b.absent_unpaired).sum()),
      "of", len(t7b), "variants; 80 +- 8% met under some pole reading in",
      int(t7b[[f"pct1000_{p}" for p in POLES]].apply(lambda r: any(abs(x - 80) <= 8 for x in r), axis=1).sum()))

**Result: blocked, methods underspecified: the two test readings disagree.** Under the main
reading the unpaired t-test finds no difference from control at 360 to 1000 V/cm at either pole
(smallest p 0.17), while the paired t-test finds the cathodic pole higher after 360 V/cm (133% of
control, +2.07% dF, p = 0.0038). The paired test finds a difference in 16 of the 18 variants and the
unpaired test in none. At 1000 V/cm the test response is 80.0% of control at the cathodic pole, 85.6%
at the anodic and 84.0% for their mean, all within 80 +- 8, and not significant (paired p 0.68 and
0.40). The paper reports its 360 V/cm cathodic point, "an accidental dF peaking to 120%", as p > 0.1,
which fits the unpaired reading; this is asked of a reader (questions/001525.yaml).

## fig7b-test-pulse-reduced-1200-1400

> Conditioning pulses of 1200 and 1400 V/cm reduce charging by the test pulse 6 us later to about
> 70% and 50% of control (p < 0.05 and p < 0.01).

Measures and tests as above: (i) paired or (ii) unpaired, pole (a) cathodic, (b) anodic, (c) mean.
Reproduced when under one combination both values are within 70 +- 7 and 50 +- 5 and the
differences reach p < 0.05 and p < 0.01.

In [ ]:
def reduced_eval(poles):
    out = {}
    for pole in POLES:
        for test in TESTS:
            v12, v14 = pct_of_control(poles, 300, pole), pct_of_control(poles, 350, pole)
            p12, p14 = test_vs_control(poles, 300, pole, test), test_vs_control(poles, 350, pole, test)
            inside = abs(v12 - 70) <= 7 and abs(v14 - 50) <= 5
            sig = p12 < 0.05 and p14 < 0.01 and v12 < 100 and v14 < 100
            out[(pole, test)] = (v12, v14, p12, p14, inside, sig)
    return out

o = reduced_eval(P0)
print("Main reading: pole, test: % of control at 1200 and 1400 V/cm, p values, within tolerance, significant")
for k, (v12, v14, p12, p14, ins, sig) in o.items():
    print(f"{k[0]:>9} {k[1]:>9}: {v12:6.1f} {v14:6.1f}  p = {p12:.4f}, {p14:.4f}  {ins} {sig}")
for pole in ("cathodic", "anodic"):
    for L in (300, 350):
        dlt = S.polarization(P0, L, pole) - S.polarization(P0, "single", pole)
        print(f"  {pole} {S.STRENGTH_V_CM[L]}: difference {dlt.mean():+.2f}% dF (s.e. {dlt.std(ddof=1)/np.sqrt(len(dlt)):.2f}), d = {dlt.mean()/dlt.std(ddof=1):+.2f}")

In [ ]:
rows = []
for v in VARIANTS:
    key, b, subs = v
    o = reduced_eval(poles_for(key, list(subs) if subs else None, b))
    full = [k for k, x in o.items() if x[4] and x[5]]
    part = [k for k, x in o.items() if x[5]]
    row = dict(variant=vname(v), reproduced_under=";".join(f"{a}/{b_}" for a, b_ in full) or "-",
               significant_under=len(part))
    for pole in POLES:
        row[f"{pole}_1200"], row[f"{pole}_1400"] = round(o[(pole, 'paired')][0], 1), round(o[(pole, 'paired')][1], 1)
    row["p1400_anod_paired"] = round(o[("anodic", "paired")][3], 4)
    row["p1200_anod_paired"] = round(o[("anodic", "paired")][2], 4)
    rows.append(row)
t7r = pd.DataFrame(rows)
print(t7r.to_string())

**Result: not reproduced.** Under the main reading the anodic pole falls to 73.3% of control
after 1200 V/cm and 50.1% after 1400 V/cm, both within tolerance, but the 1200 V/cm reduction is
not significant (paired p = 0.21, unpaired p = 0.15) and the 1400 V/cm one reaches p = 0.031 and
0.016, not p < 0.01. The cathodic pole falls further, to 22.1% and -32.9% of control (paired p =
0.045 and 0.018), and the mean of both poles to 59.3% and 27.4% (paired p = 0.039 and 0.012). No
pole and test combination meets the criterion in any of the 18 variants; in 3 the reductions reach
both significance levels with values outside tolerance (partial by the criterion). The reductions
themselves are present at every pole: the 1400 V/cm difference from control is -8.2% dF at each pole
(paired d -1.42 cathodic, -1.22 anodic).

## fig5b-conductivity-calibration

> Under Eq. (2), effective membrane conductivity barely affects charging while it stays far below
> cytosol and medium conductivity; a 10-fold increase reduces IMP by about 2% and a 100-fold
> increase by 12%.

A computation from Eq. (2) with the Methods' parameters; no recording. The equation as printed in
the PDF (page 3) matches the form the claims file reads from the text.

In [ ]:
lm0 = S.LAMBDA_M_INTACT
f0 = S.f_factor(lm0)
print(f"f for intact membrane (3e-7 S/m): {f0:.5f}")
for k in (10, 100, 1000, 1e4):
    f = S.f_factor(lm0 * k)
    print(f"{k:>7g}-fold (lm = {lm0*k*1e3:.4g} mS/m): reduction 1 - f/1.5 = {100*(1 - f/1.5):.2f}%, "
          f"relative to intact f = {100*(1 - f/f0):.2f}%")
lms = np.logspace(-7, 0, 400)
red = 100 * (1 - S.f_factor(lms) / 1.5)
fig, ax = plt.subplots(figsize=(4, 3))
ax.semilogx(lms * 1e3, 100 - red); ax.set_xlabel("lm, mS/m"); ax.set_ylabel("IMP, % of IMPC")
plt.show()
print(f"plotted IMP ranges {np.min(100 - red):.1f} to {np.max(100 - red):.1f}% of IMPC; "
      f"lm where IMP falls to 50%: {lms[np.argmin(abs(100 - red - 50))]*1e3:.3g} mS/m")
# Sensitivity of the 10-fold value to the parameters the Methods fix
for name, kw in [("R 6 um", dict(R=6e-6)), ("R 10 um", dict(R=10e-6)), ("d 7 nm", dict(d=7e-9))]:
    print(name, f"10-fold: {100*(1 - S.f_factor(lm0*10, **kw)/1.5):.2f}%, 100-fold: {100*(1 - S.f_factor(lm0*100, **kw)/1.5):.2f}%")

**Result: partially reproduced.** Eq. (2) with the Methods' parameters gives f = 1.498 for the
intact membrane, an IMP reduction of 11.87% for a 100-fold rise in membrane conductivity (within
12 +- 1.2) and 1.33% for a 10-fold rise, outside 2 +- 0.5 (1.20% relative to the intact f). Within the
ranges tried for R (6 to 10 um) the 10-fold reduction stays between 1.00% and 1.65%.

## Summary

| Claim | Outcome |
|---|---|
| fig7a-test-pulse-cosine-up-to-1000 | reproduced, resting on assumptions |
| fig7b-test-pulse-unchanged-to-760 | blocked: methods underspecified (paired and unpaired tests disagree) |
| fig7b-test-pulse-reduced-1200-1400 | not reproduced |
| fig5b-conductivity-calibration | partially reproduced |
| fig7b-conditioning-pulse-below-linear | blocked: data missing |
| fig7c-conductivity-drops-10-fold, fig7c-full-reseal-mild-residual-strong | blocked: data missing (depend on the claim above) |
| fig1, fig2 (6), fig3 (2), fig4 (6), fig5a (2), fig5c, fig6 (2): 20 claims | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The perimeter trace is found automatically, not
drawn by hand; the cell is the largest object in the field; the field direction is taken from the
data; the conditioning strengths are mapped to the files' volts by rank; the benchmark is the
dataset's single 50 V pulse, where the paper describes a 150/150 pair; six cells are used where the
paper has five; a background and the dark frames are handled under two readings each, with the
background-subtracted, dark-dropped reading adopted.

**What was found about the dataset.** It holds only the two-pulse experiment of Fig. 7 (10 cells,
of which 6 at the paper's 5-us interval; the other 4 at a 1-us interval the paper does not report).
The files give pulses in generator volts, not field strengths, and an electrode gap of 1.00 mm where
the paper gives 1.6 mm; the shared description lists field strengths (0.15 to 1.57 kV/cm) and
intervals (1, 5, 25, 125 us) that the files do not hold. The 150/150 benchmark of the paper is a
single pulse in the files. Most files have a dark first image (no laser flash), and the series
description says exposures began 1 us after the start of recording, while the images show the
pulse from image 40. No recording is deposited twice.